In [ ]:
import os
import sys
sys.path.append("..")
import utils
get_ipython().run_line_magic('matplotlib', 'inline')

import numpy as np
import pandas as pd
import anndata
import scanpy as sc

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)

# Load data

In [ ]:
fig_folder = '../../figures/spatial/islet_analysis/extra_islet/based_on_M/'
data_folder = '../../data/cell2loc_extra_islet/based_on_mercedes_annotation/'
res_folder = '../../results/spatial/islet_analysis/extra_islet/based_on_M/'

os.makedirs(fig_folder, exist_ok=True)
os.makedirs(data_folder, exist_ok=True)
os.makedirs(res_folder, exist_ok=True)

In [6]:
ct_palette = utils.load_ct_palette()
cst_palette = utils.load_cst_palette()

groups = ['ND-Lean', 'ND-Obese', 'Pre-T2D', 'T1D', 'T2D']
group_colors = {group: sns.color_palette('husl', 5)[i] 
                for i, group in enumerate(groups)}

## load spatial data

In [ ]:
## after integrating with islet-cell2location results
spatial = sc.read_h5ad('../../data/YK_raw_spatial_islet_c2l_annot_hq.h5ad')
spatial

AnnData object with n_obs × n_vars = 1390113 × 13658
    obs: 'n_counts', 'barcode', 'x', 'y', '_indices', '_scvi_batch', '_scvi_labels', 'total_abundance', 'Endocrine', 'Acinar', 'Ductal', 'Endothelial', 'Immune', 'Stellate', 'Schwann', 'max_pred', 'max_pred_celltype', 'sample', 'Endocrine_local_I', 'Endocrine_local_p', 'Endocrine_cluster', 'Acinar_local_I', 'Acinar_local_p', 'Acinar_cluster', 'Ductal_local_I', 'Ductal_local_p', 'Ductal_cluster', 'Endothelial_local_I', 'Endothelial_local_p', 'Endothelial_cluster', 'Immune_local_I', 'Immune_local_p', 'Immune_cluster', 'Stellate_local_I', 'Stellate_local_p', 'Stellate_cluster', 'Schwann_local_I', 'Schwann_local_p', 'Schwann_cluster', 'Endocrine_local_I_norm', 'Acinar_local_I_norm', 'Ductal_local_I_norm', 'Endothelial_local_I_norm', 'Immune_local_I_norm', 'Stellate_local_I_norm', 'Schwann_local_I_norm', 'endocrine', 'lisa_selected', 'endocrine_score', 'endocrine_score_norm', 'ec_selected', 'labels', 'islet', 'islets_to_keep', 'islets_in_

In [ ]:
spatial_islet = spatial[spatial.obs['islets_in_out'] == 'in'].copy()

# endocrine bead annotation after cell2location

In [ ]:
# import the saved model trained on ref data
spatial_file_low = f"{os.path.join(data_folder, 'LQ_samples/spatial_extra_islet_after_c2l_LQ.h5ad')}"
spat_low = sc.read_h5ad(spatial_file_low)
spat_low.obsm.pop('q05_cell_proportions', None)

# import the saved model trained on ref data
spatial_file_high = f"{os.path.join(data_folder, 'HQ_samples/spatial_extra_islet_after_c2l_HQ.h5ad')}"

spat_high = sc.read_h5ad(spatial_file_high)
spat_high.obsm.pop('q05_cell_proportions', None)

spat = anndata.concat([spat_low, spat_high], join="outer")

## annotations

In [27]:
from scipy.stats import binom

# ================== 0) input and cleaning ==================
type_cols = ["α","β","γ","δ"]
group_col, sample_col, islet_col = "group", "sample", "islet"

# q05 abundance matrix
abund = pd.DataFrame(
    spat.obsm["q05_cell_abundance_w_sf"].values,
    index=spat.obs_names,
    columns=type_cols
)
type_cols = [c for c in type_cols if c in abund.columns]
assert len(type_cols) == 4, "need exact four columns: α/β/γ/δ"

abund = abund[type_cols].copy()
abund = abund.join(spat.obs[[group_col, sample_col, islet_col]])
print(f"In total {abund.shape[0]} islet beads")

# 5% quantile clipping (zeroing) for each type + calculation of total endocrine volume
abund_filter = abund[type_cols].apply(lambda x: np.quantile(x, 0.05))
abund_clean = abund.copy()
abund_clean[type_cols] = abund_clean[type_cols].mask(abund_clean[type_cols] < abund_filter, 0.0)
abund_clean["endo_total_q05"] = abund_clean[type_cols].sum(axis=1)

# Low quality: The total amount is below the global 5% percentile
enable_low_quality = True
total_q05 = float(np.quantile(abund_clean["endo_total_q05"], 0.05))
low_quality = enable_low_quality & (abund_clean["endo_total_q05"] < total_q05)

# Normalization to get fraction
frac = abund_clean[type_cols].div(abund_clean["endo_total_q05"], axis=0).fillna(0.0)

# Information for ordering
dominant_type = frac.idxmax(axis=1)
dominance = frac.max(axis=1)
second_dominance = frac.apply(lambda r: r.nlargest(2).iloc[-1] if (r > 0).sum() >= 2 else 0.0, axis=1)
third_dominance  = frac.apply(lambda r: r.nlargest(3).iloc[-1] if (r > 0).sum() >= 3 else 0.0, axis=1)
fourth_dominance = frac.min(axis=1)
gap12 = dominance - second_dominance
gap23 = second_dominance - third_dominance
gap34 = third_dominance - fourth_dominance

top1_col = dominant_type
top2_col = frac.apply(lambda r: r.nlargest(2).index[1] if (r > 0).sum() >= 2 else pd.NA, axis=1)
top3_col = frac.apply(lambda r: r.nlargest(3).index[2] if (r > 0).sum() >= 3 else pd.NA, axis=1)

In total 23904 islet beads


In [28]:
# I decide not to apply these corrections because I will smooth or sharpen the data anaway

# null weight: >1 improves the expected value of this type at p0 (harder to be significant)
null_weight = {"α": 1.00, "β": 1.00, "γ": 1.00, "δ": 1.00}
# Statistic weight: <1 reduces the contribution of this type in the S_k statistic
stat_weight = {"α": 1.00, "β": 1.00, "γ": 1.00, "δ": 1.00}
# select 'null' / 'stat' / 'both' / 'none'
weight_mode = "both"

# p0 construction method：'islet' | 'global' | 'equal'
p0_mode = "equal"

# ================== 1) weights and baseline p0 ==================
def apply_null_weight(p0_series: pd.Series) -> pd.Series:
    """
    Reweight a baseline class distribution (p0_series) using 'null' priors,
    then renormalize to sum to 1. No-op unless weight_mode is 'null' or 'both'.
    """
    if weight_mode in ("null", "both"):
        # Build per-class multiplier vector from a dict `null_weight`.
        # Missing classes default to weight 1.0 (i.e., unchanged).
        w = pd.Series({k: null_weight.get(k, 1.0) for k in type_cols})
        # Elementwise scale and clip away exact zeros (stability for logs/divisions).
        p = (p0_series * w).clip(1e-12)
        # Renormalize to a proper distribution.
        return p / p.sum()
    # If not using null weights, return the baseline unchanged.
    return p0_series

# Choose the *global* baseline p0 according to your mode:
if p0_mode == "equal":
    # Uniform prior across the 4 classes (adjust 4 if len(type_cols) != 4).
    base_p0_global = pd.Series([0.25]*4, index=type_cols)
elif p0_mode == "global":
    # Empirical prior from the global mean of observed fractions,
    # normalized to sum to 1 and clipped for stability.
    base_p0_global = (frac[type_cols].mean() / frac[type_cols].mean().sum()).clip(1e-12)

# Optionally build *per-islet* baseline p0 if requested.
if p0_mode == "islet":
    # 1) Attach islet labels to rows (join) and group by islet
    p0_by_islet = frac.join(abund_clean[[islet_col]]).groupby(islet_col)[type_cols].mean()

    # 2) Turn each group's row into a valid probability vector (sum to 1),
    #    and fill missing groups with the global baseline.
    p0_by_islet = p0_by_islet.div(p0_by_islet.sum(axis=1), axis=0).fillna(base_p0_global)

    # 3) Apply null weights to each islet-specific baseline distribution.
    p0_by_islet = p0_by_islet.apply(apply_null_weight, axis=1)
else:
    # If not using per-islet baselines, keep none and apply null weights to global baseline.
    p0_by_islet = None
    base_p0_global = apply_null_weight(base_p0_global)

def get_p0(idx):
    """
    Retrieve the appropriate baseline distribution p0 for a given row/index:
    - If 'islet' mode: use that row's islet-specific baseline if available, else fallback to global.
    - Otherwise: use the global baseline.
    Always return a clipped float vector for numerical stability.
    """
    if p0_mode == "islet":
        isl = abund_clean.loc[idx, islet_col]
        p0 = p0_by_islet.loc[isl] if isl in p0_by_islet.index else base_p0_global
    else:
        p0 = base_p0_global
    return p0.astype(float).clip(1e-12)

def apply_stat_weight_vec(vec: np.ndarray) -> np.ndarray:
    """
    Optionally apply a second set of per-class weights (e.g., statistical/importance weights)
    to a vector of counts/scores, used in the permutation test statistic.
    No-op unless weight_mode is 'stat' or 'both'.
    """
    if weight_mode in ("stat", "both"):
        w = np.array([stat_weight.get(k, 1.0) for k in type_cols], dtype=float)
        return vec * w
    return vec

# ================== 2) significance test ==================
# N_eff: pseudo sample size to convert fractions into integer counts
N_eff = 100           # Larger gives finer resolution but heavier compute
n_perm = 1000         # Number of multinomial simulations for permutation p-values
rng = np.random.default_rng(123)  # Reproducible randomness

def holm_bonferroni(pvals: pd.Series) -> pd.Series:
    """
    Multiple-testing correction using Holm-Bonferroni (step-down).
    Returns adjusted p-values aligned to the original index.
    """
    # Sort p-values ascending, track order
    order = pvals.sort_values()
    m = len(order)

    # Holm adjustment in sorted order: (m - i) * p_(i)
    adj_sorted = np.array([(m - i) * p for i, p in enumerate(order.values)], dtype=float)

    # Enforce monotonicity of adjusted p-values across the sequence
    adj_sorted = np.maximum.accumulate(adj_sorted)

    # Map back to original order, cap at 1.0
    adj = pd.Series(adj_sorted, index=order.index).reindex(pvals.index)
    return adj.clip(upper=1.0)

def test_top1_holm(row_f: pd.Series, idx) -> float:
    """
    For one row of observed fractions (row_f), test the top class against its
    null probability using a binomial upper-tail test, and apply Holm correction
    across all classes for that row. Return the adjusted p-value of the top class.
    """
    # Baseline probabilities for this row (global or islet-specific)
    p0 = get_p0(idx)

    # Convert fractions into counts under N_eff
    counts = (row_f * N_eff).round().astype(int)

    # Compute one-sided binomial p-value for each class (Pr[X >= x | n, p0])
    pvals = pd.Series(index=type_cols, dtype=float)
    for t in type_cols:
        x = int(counts[t])
        p0_i = float(p0[t])
        pvals[t] = 1.0 - binom.cdf(x - 1, N_eff, p0_i)  # upper tail

    # Multiple-testing adjust, then return the adjusted p-value of the observed top-1 class
    adj = holm_bonferroni(pvals)
    top1 = row_f.idxmax()
    return float(adj[top1])

def perm_pvalue_Sk(row_f: pd.Series, idx, k=2) -> float:
    """
    Permutation-style p-value comparing the sum of the top-k (possibly reweighted) counts
    to its null distribution generated by multinomial draws under p0.

    Steps:
    - Convert row fractions to counts under N_eff
    - Optionally apply 'stat' weights
    - Define S_obs = sum of the largest k entries
    - Simulate counts from Multinomial(N_eff, p0) n_perm times
    - Compute S_sim for each simulation
    - p-value = (1 + # {S_sim >= S_obs}) / (n_perm + 1)   (add-one smoothing)
    """
    # Baseline probabilities (vector)
    p0 = get_p0(idx).values

    # Observed counts and weighted statistic
    obs_counts = (row_f.values * N_eff).round().astype(int)
    obs_w = apply_stat_weight_vec(obs_counts.astype(float))
    S_obs = np.sort(obs_w)[::-1][:k].sum()

    # Null simulations under multinomial(p0)
    sims = rng.multinomial(N_eff, p0, size=n_perm)
    sims_w = apply_stat_weight_vec(sims.astype(float))
    S_sim = np.sort(sims_w, axis=1)[:, ::-1][:, :k].sum(axis=1)

    # Add-one smoothed permutation p-value
    pval = (1 + (S_sim >= S_obs).sum()) / (n_perm + 1)
    return float(pval)

# Compute p-values row by row
top1_adj_p = pd.Series(index=frac.index, dtype=float)
S2_pval = pd.Series(index=frac.index, dtype=float)
S3_pval = pd.Series(index=frac.index, dtype=float)

for idx, row in frac[type_cols].iterrows():
    top1_adj_p[idx] = test_top1_holm(row, idx)
    S2_pval[idx] = perm_pvalue_Sk(row, idx, k=2)
    S3_pval[idx] = perm_pvalue_Sk(row, idx, k=3)

In [30]:
# ================== 3) Based on salience + Ambiguous only when "very close" ==================
alpha_top1 = 0.05
alpha_S2   = 0.05
alpha_S3   = 0.05

# "Very close" threshold (two ways to write, choose one)
close_span = 0.12   # Solution A: max-min < 0.12 is considered close (recommended, simple and robust)
# Or use small adjacent differences:
close_gap  = 0.08   # Option B: adjacent differences are all < 0.08

min_frac_3mix = 0.20

def is_very_close_row(a, b, c, d):
    # SolutionA（default）：based on max-min 
    vals = np.array([a, b, c, d], dtype=float)
    return (vals.max() - vals.min()) < close_span

    # If you want to use solution B instead, comment out the return statement above and change it to:
    # gaps_ok = (abs(a-b) < close_gap) and (abs(b-c) < close_gap) and (abs(c-d) < close_gap)
    # return gaps_ok

# Ordered naming of mixed tags
mix_type_order = ["α","β","γ","δ"]
mix_type_rank = {k: i for i, k in enumerate(mix_type_order)}
def ordered_pair(a, b):
    if pd.isna(a) or pd.isna(b): return pd.NA
    key = lambda x: mix_type_rank.get(x, 1e9)
    first, second = sorted([a, b], key=key)
    return f"{first}_{second}"
def ordered_triplet(a, b, c):
    if pd.isna(a) or pd.isna(b) or pd.isna(c): return pd.NA
    key = lambda x: mix_type_rank.get(x, 1e9)
    first, second, third = sorted([a, b, c], key=key)
    return f"{first}_{second}_{third}"

label = pd.Series(index=frac.index, dtype="object")

# 1) Label "Low quality" first 
label[low_quality] = "Low quality"
eligible = ~low_quality

# 2) Simplicity first：single --> 2-mix --> 3-mix
# 2.1 single
remaining = eligible & (label.isna())
is_single = remaining & (top1_adj_p <= alpha_top1)
label[is_single] = dominant_type[is_single]

# 2.2 2-mix
remaining = remaining & (label.isna())
in_2mix = remaining & (S2_pval <= alpha_S2)
label[in_2mix] = [
    ordered_pair(a, b)
    for a, b in zip(top1_col[in_2mix], top2_col[in_2mix])
]

# 2.1 3-mix
remaining = remaining & (label.isna())
in_3mix = remaining & (S3_pval <= alpha_S3)
label[in_3mix] = [
    ordered_triplet(a, b, c)
    for a, b, c in zip(top1_col[in_3mix], top2_col[in_3mix], top3_col[in_3mix])
]

# 3) For those who have not yet marked it: mark it as Ambiguous only when "the four types are very close", otherwise mark it as Single (to prevent Ambiguous from exploding)
remaining = eligible & (label.isna())
if remaining.any():
    # Determine whether it is "very close" line by line
    close_mask = []
    for idx in remaining[remaining].index:
        a = dominance.loc[idx]
        b = second_dominance.loc[idx]
        c = third_dominance.loc[idx]
        d = fourth_dominance.loc[idx]
        close_mask.append(is_very_close_row(a, b, c, d))
    close_mask = pd.Series(close_mask, index=remaining[remaining].index)

    # very close -> Ambiguous
    amb_idx = close_mask[close_mask].index
    label.loc[amb_idx] = "α_β_γ_δ"
    non_close_idx = close_mask[~close_mask].index

In [31]:
remaining_nc = pd.Index(non_close_idx)

# ----- non-close resolution: prefer 3-mix vs 2-mix by p-values + a clear post-k gap -----
if len(remaining_nc):
    
    # 2-mix if S2 wins AND drop to #3 is clear (gap23 >= min_gap)
    cond_s2_loose = (S3_pval.loc[remaining_nc] > S2_pval.loc[remaining_nc])

    idx_2mix_nc = remaining_nc[cond_s2_loose.fillna(False)]
    if len(idx_2mix_nc):
        label.loc[idx_2mix_nc] = [
            ordered_pair(a, b)
            for a, b in zip(top1_col.loc[idx_2mix_nc],
                            top2_col.loc[idx_2mix_nc])
        ]
        
    # remaining is 3-mix assignment
    idx_3mix_nc = remaining_nc.difference(idx_2mix_nc)

    if len(idx_3mix_nc):
        label.loc[idx_3mix_nc] = [
            ordered_triplet(a, b, c)
            for a, b, c in zip(top1_col.loc[idx_3mix_nc],
                               top2_col.loc[idx_3mix_nc],
                               top3_col.loc[idx_3mix_nc])
        ]

In [32]:
# ================== 4) summary ==================
annot = pd.DataFrame({
    "group": abund_clean[group_col],
    "sample": abund_clean[sample_col],
    "islet":  abund_clean[islet_col],
    "endo_total_q05": abund_clean["endo_total_q05"],
    "label": label,
    "top1_adj_p": top1_adj_p,
    "S2_pval": S2_pval,
    "S3_pval": S3_pval,
    "dominant": dominant_type,
}, index=frac.index)

for ct in type_cols:
    annot[f"{ct}_abs"]  = abund_clean[ct]
    annot[f"{ct}_frac"] = frac[ct]

print("\n--- Low-quality threshold (q05 on total) ---")
print(f"{'enabled' if enable_low_quality else 'disabled'}; cutoff={total_q05:.6f}")
print("\n--- Weight modes ---")
print(f"p0_mode={p0_mode}, weight_mode={weight_mode}")
print("null_weight=", null_weight)
print("stat_weight=", stat_weight)
print("\n--- Label counts ---")
print(annot["label"].value_counts(dropna=False).sort_index().to_string())

# add the cell2location to spatial_islet
for col in ['α_frac', 'β_frac', 'γ_frac', 'δ_frac', 'endo_total_q05', 'label',]:
    spat.obs.loc[annot.index, col] = annot[col]


--- Low-quality threshold (q05 on total) ---
enabled; cutoff=0.142505

--- Weight modes ---
p0_mode=equal, weight_mode=both
null_weight= {'α': 1.0, 'β': 1.0, 'γ': 1.0, 'δ': 1.0}
stat_weight= {'α': 1.0, 'β': 1.0, 'γ': 1.0, 'δ': 1.0}

--- Label counts ---
label
Low quality    1196
α              7824
α_β            2398
α_β_γ           871
α_β_γ_δ        4477
α_β_δ           813
α_γ            1032
α_γ_δ           402
α_δ             642
β              2257
β_γ             169
β_γ_δ            42
β_δ             219
γ               797
γ_δ             197
δ               568


## save data

In [ ]:
spat.write(os.path.join(data_folder, 'spatial_extra_islet_after_cell2loc_annotation_based_on_M.h5ad'))
spatial_islet.write('../../data/cell2loc_islet/spatial_islet_after_cell2loc_annotation.h5ad')

In [119]:
src = spat.obsm['q05_cell_abundance_w_sf']          # DataFrame (rows = islet_after.obs_names)
filled = src.reindex(index=spatial.obs_names, fill_value=np.nan)  # or fill_value=np.nan
spatial.obsm['q05_cell_abundance_w_sf_extra_islet'] = filled

In [124]:
for col in ['α_frac', 'β_frac', 'γ_frac', 'δ_frac', 'endo_total_q05']:
    spatial.obs.loc[spatial_islet.obs_names, col] = spatial_islet.obs[col]
    spatial.obs.loc[spat.obs_names, col] = spat.obs[col]

In [126]:
cols = {
    'islet_label': 'label',
    'islet_label_smooth': 'dominant_smooth',
    'islet_label_sharpen': 'dominant_sharpen',
}

# safety: make sure the indices align as expected
assert set(spat.obs_names).issubset(set(spatial.obs_names))

for dst, src in cols.items():
    # ensure destination col exists and is string-typed
    spatial.obs[dst] = spatial.obs.get(dst, pd.Series(index=spatial.obs.index, dtype="string")).astype("string")
    # assign values (as string to avoid categorical clashes)
    spatial.obs.loc[spat.obs_names, dst] = spat.obs[src].astype("string")

In [129]:
spatial.obs['Endocrine_type'] = 'no'
spatial.obs.loc[spatial.obs['islets_in_out'] == 'in', 'Endocrine_type'] = 'islet_endocrine'
spatial.obs.loc[spat.obs_names, 'Endocrine_type'] = 'extra_endocrine'

In [ ]:
anndata.settings.allow_write_nullable_strings = True
spatial.write('../../data/YK_raw_spatial_islet_extra_islet_c2l_annot_hq.h5ad')